In [1]:
import plotly
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import prep_utils

import utils_for_plotly_rewrites
import helper_utils

In [2]:
HE_SHEET = "he test.xlsx"
df = pd.read_excel(HE_SHEET)
df_filtered, should_exit_early = prep_utils.prep_other(data_df = df, detect_columns=None, nm_col=None, int_col=None)


(prep_other) ymax =  247.148


In [3]:
def axis_labels(
    plot_title=None,
    x_title=utils_for_plotly_rewrites.X_TITLE,
    y_title=utils_for_plotly_rewrites.Y_TITLE,
    x_range=[utils_for_plotly_rewrites.X_MIN, utils_for_plotly_rewrites.X_MAX],
    y_range=None, 
    show_grid=True,
    bg_colour = utils_for_plotly_rewrites.BG,
    text_colour=utils_for_plotly_rewrites.COLOUR,
    grid_color='#333333',
    grid_width=0.1,
    grid_dash='dot',
    line_color='#333333',
    line_width=0.2,
    dtick_x=50, 
    dtick_y=50, 
    fig_width=utils_for_plotly_rewrites.fig_size[0] * 70, # will have to keep adjusting
    fig_height=utils_for_plotly_rewrites.fig_size[1] * 80 
):

    layout_config = dict(
        plot_bgcolor=bg_colour,
        paper_bgcolor=bg_colour,
        font=dict(color=text_colour),
        width=fig_width,
        height=fig_height,
        xaxis=dict(
            title=x_title,
            range=x_range,
            showgrid=show_grid,
            gridcolor=grid_color,
            gridwidth=grid_width,
            griddash=grid_dash,
            showline=True,
            linecolor=line_color,
            linewidth=line_width,
            mirror=True,
            tickfont=dict(color=text_colour),
            dtick=dtick_x
        ),
        yaxis=dict(
            title=y_title,
            range=y_range, # Will be None if not specified, allowing auto-scale
            showgrid=show_grid,
            gridcolor=grid_color,
            gridwidth=grid_width,
            griddash=grid_dash,
            showline=True,
            linecolor=line_color,
            linewidth=line_width,
            mirror=True,
            tickfont=dict(color=text_colour),
            dtick=dtick_y
        ),
        # Add a title if provided
        title=dict(
            text=plot_title,
            font=dict(color=text_colour) if plot_title else None 
        ) if plot_title else None
    )

    if not show_grid:
        layout_config['xaxis']['showgrid'] = False
        layout_config['yaxis']['showgrid'] = False

    return layout_config


In [38]:
def peak_labels(data_df, show_label_colour, has_any_nist):

    if has_any_nist is False:
        peaks, _ = find_peaks(data_df[helper_utils.INT_col], prominence=utils_for_plotly_rewrites.dynamic_prominence(helper_utils.DEFAULT_PROM_PERC, prep_utils.int_range)) 
        int_col = helper_utils.INT_col
    else:
        peaks, _ = find_peaks(data_df['_raw_int'], prominence=helper_utils.DEFAULT_PROM_PERC)
        int_col = ['_raw_int']

    labels = []
    for peak_index in peaks:
        row = data_df.iloc[peak_index]
        wavelength = row[helper_utils.wl_col]
        #wl_labels = float(row)
        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)
        final_int_scale = 1.0
        colour_rgb=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_int_scale)
        color_str = f"rgb({int(colour_rgb[0])}, {int(colour_rgb[1])}, {int(colour_rgb[2])})"
        if show_label_colour is True:
            font_colour = utils_for_plotly_rewrites.COLOUR
        else:
            font_colour = color_str
        label_config = dict(
            x=row[helper_utils.wl_col],
            y=row[int_col],
            text=f"{row[helper_utils.wl_col]:.2f} nm",
            showarrow=False,
            yshift=15,
            opacity=1,
            font=dict(color=font_colour)
        )
        labels.append(label_config)

    return labels

In [36]:
def reg_peak_labels(data_df, has_any_nist):
    if has_any_nist is False:
        peaks, _ = find_peaks(data_df[helper_utils.INT_col], prominence=utils_for_plotly_rewrites.dynamic_prominence(helper_utils.DEFAULT_PROM_PERC, prep_utils.int_range)) 
        int_col = helper_utils.INT_col
    else:
        peaks, _ = find_peaks(data_df['_raw_int'], prominence=helper_utils.DEFAULT_PROM_PERC)
        int_col = ['_raw_int']

    labels = []
    for peak_index in peaks:
        row = df_filtered.iloc[peak_index]
        label_config = dict(
            x=row[helper_utils.wl_col],
            y=row[int_col],
            text=f"{row[helper_utils.wl_col]:.2f} nm",
            showarrow=False,
            yshift=15,
            opacity=1,
            font=dict(color=utils_for_plotly_rewrites.COLOUR) # Text color for annotation
        )
        labels.append(label_config)
    return labels

def colour_labels(df_filtered):
    peaks, _ = find_peaks(df_filtered['Grey Val'], prominence=utils_for_plotly_rewrites.dynamic_prominence(utils_for_plotly_rewrites.prominence, int_range))

    labels = []
    for peak_index in peaks:
        row = df_filtered.iloc[peak_index]
        wavelength = row['nm']
        #wl_labels = float(row)
        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)
        final_int_scale = 1.0
        colour_rgb=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_int_scale)
        color_str = f"rgb({int(colour_rgb[0]*255)}, {int(colour_rgb[1]*255)}, {int(colour_rgb[2]*255)})"
        label_config = dict(
            x=row['nm'],
            y=row['Grey Val'],
            text=f"{row['nm']:.2f} nm",
            showarrow=False,
            yshift=15,
            opacity=1,
            font=dict(color=color_str, ) # Text color for annotation
        )
        labels.append(label_config)
    return labels

In [39]:
def line_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    line_fig = go.Figure()

    if show_peak_labels is True:
        line_fig.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))


    for i in range(len(data_df) - 1):
        wavelength_start = float(data_df.iloc[i][helper_utils.wl_col])
        wavelength_end = float(data_df.iloc[i+1][helper_utils.wl_col])
        int_factor = float(data_df.iloc[i]['Norm_Int'])
        base_rgb_val = utils_for_plotly_rewrites.rgb(wavelength_start)

        if scale_by_int is True:
            final_intensity_scale = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, int_factor)
        else:
            final_intensity_scale = 1.0

        color_rgb_float = utils_for_plotly_rewrites.colored_rgb(base_rgb_val, final_intensity_scale)
        color_str = f"rgb({int(color_rgb_float[0])}, {int(color_rgb_float[1])}, {int(color_rgb_float[2])})"

        line_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[data_df.iloc[i][helper_utils.INT_col], data_df.iloc[i+1][helper_utils.INT_col]],
            mode='lines',
            line=dict(color=color_str, width=2), # linewidth=2 from line_plot_iteration
            showlegend=False
        ))

        #print(color_str)

    line_fig.update_layout(axis_labels())
    line_fig.show()

In [40]:
line_iter(data_df=df_filtered, show_peak_labels=True, show_label_colour=False, scale_by_int=True)

In [7]:
def scatter_iter(df_filtered):
    alpha_factor = df_filtered['Normalized_int']
    base_marker_size_plotly = 2
    max_marker_size_factor_plotly = 8
    sizes = base_marker_size_plotly + (max_marker_size_factor_plotly * alpha_factor)
    alphas = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha_scatter, alpha_factor)

    rgba_colors = []
    for i in range(len(df_filtered)):
        wavelength = df_filtered.iloc[i]['nm']
        r, g, b = utils_for_plotly_rewrites.rgb(wavelength, gamma=utils_for_plotly_rewrites.gamma_factor)
        a = alphas.iloc[i]
        rgba_colors.append(f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})")

    scatter_fig=go.Figure()
    scatter_fig.add_trace(go.Scatter(
        x=df_filtered['nm'],
        y=df_filtered['Grey Val'],
        mode='markers',
        marker=dict(
            color=rgba_colors,
            size=sizes,
            line=dict(
                width=0
            )
        ),
        showlegend=False,
    ))

    scatter_fig.update_layout(axis_labels())
    scatter_fig.show()

In [10]:
def gaussian_iter(df_filtered):
    # Detect peaks
    dyn_prominence = utils_for_plotly_rewrites.dynamic_prominence(0.08, prep_utils.int_range)
    peaks_indices, properties = find_peaks(df_filtered['Grey Val'], prominence=dyn_prominence)

    # Create a new, denser wavelength array for plotting the synthetic spectrum
    x_synthetic = np.linspace(400, 750, 1000) # 1000 points for a smooth synthetic curve
    y_synthetic = np.zeros_like(x_synthetic)

    for i, peak_idx in enumerate(peaks_indices):
        peak_nm = df_filtered.iloc[peak_idx]['nm']
        peak_amplitude = df_filtered.iloc[peak_idx]['Grey Val']
        normalized_amplitude = df_filtered.iloc[peak_idx]['Normalized_int']

        # Scale sigma based on normalized intensity (higher intensity = broader peak)
        sigma = utils_for_plotly_rewrites.base_sigma_nm + (utils_for_plotly_rewrites.max_sigma_multiplier - 1) * utils_for_plotly_rewrites.base_sigma_nm * normalized_amplitude

        # Create a Gaussian curve for this peak
        gaussian_curve = peak_amplitude * np.exp(-((x_synthetic - peak_nm)**2) / (2 * sigma**2))
        y_synthetic += gaussian_curve # Add to the total synthetic spectrum

    # Normalize the synthetic spectrum intensities for coloring
    min_y_synthetic = y_synthetic.min()
    max_y_synthetic = y_synthetic.max()
    if (max_y_synthetic - min_y_synthetic) == 0:
        normalized_y_synthetic = np.ones_like(y_synthetic)
    else:
        normalized_y_synthetic = (y_synthetic - min_y_synthetic) / (max_y_synthetic - min_y_synthetic)

    gauss_fig=go.Figure()

    # Iterate through each segment of the synthetic spectrum to apply color and alpha
    for i in range(len(x_synthetic) - 1):
        wavelength_start = x_synthetic[i]
        wavelength_end = x_synthetic[i+1]

        y_val_start = y_synthetic[i]
        y_val_end = y_synthetic[i+1]

        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=utils_for_plotly_rewrites.gamma_factor)

        segment_alpha = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha, normalized_y_synthetic[i])

        r, g, b = int(base_rgb_tuple[0]*255), int(base_rgb_tuple[1]*255), int(base_rgb_tuple[2]*255)
        fill_color_str = f"rgba({r}, {g}, {b}, {segment_alpha})"
        # Changed line_color_str to a uniform subtle grey and width to 0 to make the filled area seamless
        line_color_str = '#333333' # Using the subtle grid color for consistency

        gauss_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[y_val_start, y_val_end],
            mode='lines',
            line=dict(color=fill_color_str, width=0), # Set width to 0 to remove segment outlines
            fill='tozeroy',
            fillcolor=fill_color_str,
            showlegend=False,
            name=f''        
        ))

    gauss_fig.update_layout(axis_labels())
    gauss_fig.show()

In [9]:
gaussian_iter(df_filtered=df_filtered)

NameError: name 'int_range' is not defined

In [ ]:
def filled_iter(df_filtered):
    filled_fig=go.Figure()
    for i in range(len(df_filtered) - 1):
        wavelength_start = df_filtered.iloc[i]['nm']
        wavelength_end = df_filtered.iloc[i+1]['nm']

        y_val_start = df_filtered.iloc[i]['Grey Val']
        y_val_end = df_filtered.iloc[i+1]['Grey Val']

        # Calculate color based on wavelength_start (as in Matplotlib version)
        # rgb, gamma_factor are expected to be defined in the global scope.
        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=utils_for_plotly_rewrites.gamma_factor) # rgb returns (R,G,B) 0-1

        # Calculate alpha for fill
        # min_alpha, Normalized_int, final_scale are expected to be defined in the global scope.
        alpha_factor = df_filtered.iloc[i]['Normalized_int']
        segment_alpha = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha, alpha_factor) # min_alpha is 0.1

        # Convert base_rgb_tuple to rgba string for fillcolor
        r, g, b = int(base_rgb_tuple[0]*255), int(base_rgb_tuple[1]*255), int(base_rgb_tuple[2]*255)
        fill_color_str = f"rgba({r}, {g}, {b}, {segment_alpha})"

        print(base_rgb_tuple)

        # Convert base_rgb_tuple to rgb string for line color
        line_color_str = f"rgb({r}, {g}, {b})"

        filled_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[y_val_start, y_val_end],
            mode='lines',
            line=dict(color=line_color_str, width=2), # linewidth=2 from line_plot_iteration
            fill='tozeroy', # Fill area below the line to y=0
            fillcolor=fill_color_str,
            showlegend=False,
            name=f'Segment {i}' # Add a name for potential debugging, though not shown
        ))

    filled_fig.update_layout(axis_labels())
    filled_fig.show()

filled_iter(df_filtered=df_filtered)

(0.43603228940359945, 0.0, 0.6148327145280073)
(0.43414613954118325, 0.0, 0.626675307094915)
(0.4317711750698661, 0.0, 0.6384622092949674)
(0.42890536592270334, 0.0, 0.6501949564723505)
(0.42554622866472513, 0.0, 0.6618750081063409)
(0.4216908042204148, 0.0, 0.6735037531445144)
(0.41733563058013384, 0.0, 0.685082514854417)
(0.4124767098098739, 0.0, 0.6966125552464034)
(0.4071094685110708, 0.0, 0.7080950791136044)
(0.401228710656141, 0.0, 0.719531237729232)
(0.39482856144640227, 0.0, 0.7309221322364995)
(0.38790240048232427, 0.0, 0.7422688167621798)
(0.3804427820742691, 0.0, 0.7535723012811691)
(0.3724413399166623, 0.0, 0.7648335542562504)
(0.3638886725453115, 0.0, 0.776053505074497)
(0.3547742049175493, 0.0, 0.7872330462993662)
(0.34508601998233374, 0.0, 0.7983730357554399)
(0.3348106520696453, 0.0, 0.8094742984609422)
(0.3239328310624941, 0.0, 0.8205376284215603)
(0.3124351622119324, 0.0, 0.8315637902976875)
(0.30029772046578135, 0.0, 0.8425535209559638)
(0.2874975292452077, 0.0, 0.85

In [ ]:
import plotly.graph_objects as go

def bar_iter(df=df_filtered):
    bar_colors = []
    for index, row in df.iterrows():
        wavelength = row['nm']
        normalized_intensity = row['Normalized_int']

        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)
        final_intensity_scale=utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, normalized_intensity) # Using global min_bright

        color_rgb_float=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_intensity_scale)
        
        # Convert float RGB (0-1) to string 'rgba(R,G,B,A)' with integer values (0-255)
        r, g, b = int(color_rgb_float[0]*255), int(color_rgb_float[1]*255), int(color_rgb_float[2]*255)
        bar_colors.append(f"rgba({r}, {g}, {b}, {1.0})")

    fig_bar = go.Figure();

    fig_bar.add_trace(go.Bar(
        x=df_filtered['nm'],
        y=df_filtered['Grey Val'],
        marker=dict(color=bar_colors), # Use marker dict for color
        marker_line_color=bar_colors,
        marker_line_width=0,
        showlegend=False,
    ));

    fig_bar.update_layout(axis_labels())
    fig_bar.show()

bar_iter()

In [ ]:
def non_rgb_iter(df_filtered=df_filtered):
    non_rgb_fig=go.Figure()

    non_rgb_fig = px.line(df_filtered, x='nm', y='Grey Val',
        title='Grey Val vs. nm (nm range 400-750) with Sharp Peak Labels',
        color_discrete_sequence=['#1f77b4']) # Matching Matplotlib's default blue

    peaks, _ = find_peaks(df_filtered['Grey Val'], prominence=utils_for_plotly_rewrites.dynamic_prominence(utils_for_plotly_rewrites.prominence, prep_utils.int_range))

    non_rgb_fig.update_layout(axis_labels(), annotations=colour_labels(df_filtered=df_filtered))
    non_rgb_fig.show()

non_rgb_iter()

In [ ]:
gaussian_iter(df_filtered=df_filtered)
scatter_iter(df_filtered=df_filtered)
line_iter(df_filtered=df_filtered)
filled_iter(df_filtered=df_filtered)

(0.43603228940359945, 0.0, 0.6148327145280073)
(0.43414613954118325, 0.0, 0.626675307094915)
(0.4317711750698661, 0.0, 0.6384622092949674)
(0.42890536592270334, 0.0, 0.6501949564723505)
(0.42554622866472513, 0.0, 0.6618750081063409)
(0.4216908042204148, 0.0, 0.6735037531445144)
(0.41733563058013384, 0.0, 0.685082514854417)
(0.4124767098098739, 0.0, 0.6966125552464034)
(0.4071094685110708, 0.0, 0.7080950791136044)
(0.401228710656141, 0.0, 0.719531237729232)
(0.39482856144640227, 0.0, 0.7309221322364995)
(0.38790240048232427, 0.0, 0.7422688167621798)
(0.3804427820742691, 0.0, 0.7535723012811691)
(0.3724413399166623, 0.0, 0.7648335542562504)
(0.3638886725453115, 0.0, 0.776053505074497)
(0.3547742049175493, 0.0, 0.7872330462993662)
(0.34508601998233374, 0.0, 0.7983730357554399)
(0.3348106520696453, 0.0, 0.8094742984609422)
(0.3239328310624941, 0.0, 0.8205376284215603)
(0.3124351622119324, 0.0, 0.8315637902976875)
(0.30029772046578135, 0.0, 0.8425535209559638)
(0.2874975292452077, 0.0, 0.85